# Módulos estándar · math y statistics

Cálculo numérico con la biblioteca estándar: redondeos, trigonometría, comparación de reales, logaritmos, enteros grandes con `isqrt` y `prod`, combinatoria, estadística descriptiva y la distribución normal con `statistics`, un generador pseudoaleatorio propio, `random` y `secrets`, y una fórmula de gestión de inventario. El último ejercicio deriva e integra un polinomio con `sympy`. Ideas de [PYnative](https://pynative.com/python-exercises-with-solutions/), [machinelearningplus](https://www.machinelearningplus.com/) y [retosdeprogramacion.com](https://retosdeprogramacion.com/ejercicios).

## 1. Redondeos y números negativos

Mostrar en una tabla `ceil`, `floor`, `trunc`, `round` e `int` para valores positivos y negativos terminados en ,5 y en ,7. Explicar el redondeo del banquero de `round`, por qué `round(2.675, 2)` da 2,67 y escribir `redondeo_escolar(x, decimales)` con `Decimal`, que redondee los ,5 alejándose del cero.

In [1]:
import math
from decimal import ROUND_HALF_UP, Decimal

print(f"{'x':>5} {'ceil':>5} {'floor':>6} {'trunc':>6} {'round':>6} {'int':>4}")
for x in (2.5, -2.5, 3.7, -3.7, 0.5, -0.5, 1.5):
    print(f"{x:>5} {math.ceil(x):>5} {math.floor(x):>6} {math.trunc(x):>6} {round(x):>6} {int(x):>4}")
    assert math.trunc(x) == int(x) == (math.floor(x) if x >= 0 else math.ceil(x))
    assert math.floor(x) == x // 1

    x  ceil  floor  trunc  round  int
  2.5     3      2      2      2    2
 -2.5    -2     -3     -2     -2   -2
  3.7     4      3      3      4    3
 -3.7    -3     -4     -3     -4   -3
  0.5     1      0      0      0    0
 -0.5     0     -1      0      0    0
  1.5     2      1      1      2    1


`round` lleva los empates al **par** más cercano (redondeo del banquero), para que al redondear muchos datos los errores no se acumulen siempre hacia arriba. Y 2,675 no se puede guardar exactamente en binario: el `float` real es un poco menor.

In [2]:
print("round de 0,5 a 4,5:", [round(x) for x in (0.5, 1.5, 2.5, 3.5, 4.5)])
print("round(2.675, 2) =", round(2.675, 2), "porque 2.675 es en realidad", Decimal(2.675))


def redondeo_escolar(x, decimales=0):
    # Decimal(str(x)) parte del texto "2.675", no del float binario
    return Decimal(str(x)).quantize(Decimal(1).scaleb(-decimales), rounding=ROUND_HALF_UP)


print("Escolar:", [str(redondeo_escolar(x)) for x in (0.5, 1.5, 2.5, -2.5)], redondeo_escolar(2.675, 2))
assert [round(x) for x in (0.5, 1.5, 2.5, 3.5)] == [0, 2, 2, 4]
assert redondeo_escolar(2.5) == 3 and redondeo_escolar(-2.5) == -3 and redondeo_escolar(2.675, 2) == Decimal("2.68")

round de 0,5 a 4,5: [0, 2, 2, 4, 4]
round(2.675, 2) = 2.67 porque 2.675 es en realidad 2.67499999999999982236431605997495353221893310546875
Escolar: ['1', '2', '3', '-3'] 2.68


## 2. `hypot`, grados y radianes

Una carretera sube 120 m mientras avanza 1,5 km en horizontal: calcular la distancia recorrida con `math.hypot`, la pendiente en % y el ángulo en grados con `atan2`. Mostrar después una tabla de ángulos notables en grados y radianes con su seno y coseno, y comprobar que $\sin^2 + \cos^2 = 1$.

In [3]:
avance, subida = 1_500, 120
print(f"Distancia: {math.hypot(avance, subida):.1f} m; pendiente: {subida / avance:.0%}; "
      f"ángulo: {math.degrees(math.atan2(subida, avance)):.2f}°")
assert math.hypot(3, 4) == 5 and math.hypot(2, 3, 6) == 7

print(f"\n{'grados':>6} {'radianes':>9} {'seno':>7} {'coseno':>7}")
for grados in (0, 30, 45, 60, 90, 180):
    r = math.radians(grados)
    print(f"{grados:>6} {r:>9.4f} {math.sin(r):>7.4f} {math.cos(r):>7.4f}")
    assert math.isclose(math.sin(r) ** 2 + math.cos(r) ** 2, 1)
    assert math.isclose(math.degrees(r), grados)
print("\ncos(90°) sin formatear:", math.cos(math.radians(90)))
assert math.radians(180) == math.pi and math.isclose(math.sin(math.radians(30)), 0.5)

Distancia: 1504.8 m; pendiente: 8%; ángulo: 4.57°

grados  radianes    seno  coseno
     0    0.0000  0.0000  1.0000
    30    0.5236  0.5000  0.8660
    45    0.7854  0.7071  0.7071
    60    1.0472  0.8660  0.5000
    90    1.5708  1.0000  0.0000
   180    3.1416  0.0000 -1.0000

cos(90°) sin formatear: 6.123233995736766e-17


## 3. `math.isclose` frente a `==`

Comprobar que `0.1 + 0.2 == 0.3` es falso y que `math.isclose` lo resuelve, que sumar diez veces 0,1 en un bucle no da exactamente 1 (`math.fsum` y, desde Python 3.12, `sum` compensan el error), y ver cuándo hace falta `abs_tol` (comparaciones con cero) y por qué una tolerancia absoluta fija no sirve para números grandes.

In [4]:
print("0.1 + 0.2 == 0.3:", 0.1 + 0.2 == 0.3, "| isclose:", math.isclose(0.1 + 0.2, 0.3))
total = 0.0
for _ in range(10):
    total += 0.1
print("Bucle con diez 0.1:", total, "| sum:", sum([0.1] * 10), "| fsum:", math.fsum([0.1] * 10))
print("isclose(1e-12, 0):", math.isclose(1e-12, 0), "| con abs_tol=1e-9:", math.isclose(1e-12, 0, abs_tol=1e-9))


def iguales_absoluto(a, b, tol=1e-9):
    return abs(a - b) < tol


a, b = 1e20, 1e20 + 1e5
print(f"1e20 y 1e20 + 1e5 -> absoluto: {iguales_absoluto(a, b)}, isclose (relativo): {math.isclose(a, b)}")
assert not 0.1 + 0.2 == 0.3 and math.isclose(0.1 + 0.2, 0.3)
assert total != 1.0 and math.fsum([0.1] * 10) == 1.0 and math.isclose(total, 1.0)
assert math.isclose(1000, 1001, rel_tol=1e-3) and not math.isclose(1, 2, rel_tol=1e-3)

0.1 + 0.2 == 0.3: False | isclose: True
Bucle con diez 0.1: 0.9999999999999999 | sum: 1.0 | fsum: 1.0
isclose(1e-12, 0): False | con abs_tol=1e-9: True
1e20 y 1e20 + 1e5 -> absoluto: False, isclose (relativo): True


## 4. Fibonacci con la fórmula de Binet

La fórmula de Binet da $F_n = \dfrac{\varphi^n - \psi^n}{\sqrt 5}$, con $\varphi = \frac{1 + \sqrt 5}{2}$ (la razón áurea) y $\psi = \frac{1 - \sqrt 5}{2}$; como $|\psi^n| / \sqrt 5 < \frac12$, basta redondear $\varphi^n / \sqrt 5$. Compararla con la versión iterativa, encontrar el primer $n$ en el que deja de ser exacta con `float` y comprobar que $F_{n+1} / F_n$ tiende a $\varphi$.

In [5]:
from itertools import count

FI = (1 + math.sqrt(5)) / 2


def fib_binet(n):
    return round(FI**n / math.sqrt(5))


def fib_iterativo(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


for n in (10, 30, 50, 70):
    print(f"F({n}) = {fib_iterativo(n)}  Binet: {fib_binet(n)}")
primero_mal = next(n for n in count() if fib_binet(n) != fib_iterativo(n))
print(f"Primer fallo en n = {primero_mal}: Binet da {fib_binet(primero_mal)}, el valor real es {fib_iterativo(primero_mal)}")
print(f"F(41)/F(40) = {fib_iterativo(41) / fib_iterativo(40):.12f}; φ = {FI:.12f}")

assert all(fib_binet(n) == fib_iterativo(n) for n in range(primero_mal))
# El error relativo de φⁿ crece con n hasta que el redondeo ya no lo absorbe
assert math.isclose(fib_binet(primero_mal), fib_iterativo(primero_mal), rel_tol=1e-12)
assert math.isclose(fib_iterativo(41) / fib_iterativo(40), FI)

F(10) = 55  Binet: 55
F(30) = 832040  Binet: 832040
F(50) = 12586269025  Binet: 12586269025
F(70) = 190392490709135  Binet: 190392490709135
Primer fallo en n = 71: Binet da 308061521170130, el valor real es 308061521170129
F(41)/F(40) = 1.618033988750; φ = 1.618033988750


## 5. Exponenciales y logaritmos

Comprobar que `exp` y `log` son inversas, comparar `log(x, 10)` con `log10(x)`, contar los dígitos de un número con `log10` (y encontrar dónde falla), relacionar `log2` con `int.bit_length`, calcular en cuántos años se duplica un capital al 3 % y ver para qué sirve `log1p`.

In [6]:
assert all(math.isclose(math.exp(math.log(x)), x) for x in (0.001, 1, 2.5, 1e10))
print("log(1000, 10) =", math.log(1000, 10), "| log10(1000) =", math.log10(1000))


def digitos(n):
    return math.floor(math.log10(n)) + 1


print("Dígitos de 2**1000:", digitos(2**1000), "=", len(str(2**1000)))
n = 10**16 - 1
print(f"Dígitos de 10**16 - 1: con log10 {digitos(n)}, con str {len(str(n))} (log10 redondea a {math.log10(n)})")

print("log2(2**50) =", math.log2(2**50), "| (2**50).bit_length() - 1 =", (2**50).bit_length() - 1)

años = math.log(2) / math.log(1.03)
print(f"Al 3 % el capital se duplica en {años:.2f} años (regla del 72: {72 / 3:.0f})")
print("log(1 + 1e-17) =", math.log(1 + 1e-17), "| log1p(1e-17) =", math.log1p(1e-17))

assert digitos(2**1000) == len(str(2**1000)) == 302 and digitos(n) != len(str(n))
assert math.isclose(1.03**años, 2)
assert all(math.log2(2**k) == k == (2**k).bit_length() - 1 for k in range(64))

log(1000, 10) = 2.9999999999999996 | log10(1000) = 3.0
Dígitos de 2**1000: 302 = 302
Dígitos de 10**16 - 1: con log10 17, con str 16 (log10 redondea a 16.0)
log2(2**50) = 50.0 | (2**50).bit_length() - 1 = 50
Al 3 % el capital se duplica en 23.45 años (regla del 72: 24)
log(1 + 1e-17) = 0.0 | log1p(1e-17) = 1e-17


## 6. Grados, minutos y segundos

Escribir `a_gms(grados_decimales)`, que devuelva el signo, los grados, los minutos y los segundos, `texto_coordenadas(lat, lon)`, que escriba una coordenada con N/S y E/O, y `a_decimal(g, m, s, hemisferio)`, la conversión inversa. Probarlas con unas coordenadas aproximadas del pico del Teide y comprobar la ida y vuelta con 1000 coordenadas aleatorias.

In [7]:
import random


def a_gms(decimal):
    signo = -1 if decimal < 0 else 1
    minutos, segundos = divmod(abs(decimal) * 3600, 60)
    grados, minutos = divmod(int(minutos), 60)
    return signo, grados, minutos, segundos


def texto_coordenadas(lat, lon):
    partes = []
    for valor, letras in ((lat, "NS"), (lon, "EO")):
        signo, g, m, s = a_gms(valor)
        partes.append(f"{g}°{m:02d}′{s:04.1f}″ {letras[signo < 0]}")
    return ", ".join(partes)


def a_decimal(grados, minutos, segundos, hemisferio):
    valor = grados + minutos / 60 + segundos / 3600
    return -valor if hemisferio in "SO" else valor


teide = (28.2724, -16.6425)
print("Teide:", teide, "->", texto_coordenadas(*teide))
print("De vuelta:", round(a_decimal(28, 16, 20.64, "N"), 4), round(a_decimal(16, 38, 33.0, "O"), 4))
assert a_gms(teide[1])[:3] == (-1, 16, 38)

rng = random.Random(2026)
for _ in range(1_000):
    lat, lon = rng.uniform(-90, 90), rng.uniform(-180, 180)
    for valor, letras in ((lat, "NS"), (lon, "EO")):
        signo, g, m, s = a_gms(valor)
        assert 0 <= m < 60 and 0 <= s < 60
        assert math.isclose(a_decimal(g, m, s, letras[signo < 0]), valor, abs_tol=1e-9)

Teide: (28.2724, -16.6425) -> 28°16′20.6″ N, 16°38′33.0″ O
De vuelta: 28.2724 -16.6425


## 7. Cuadrados perfectos y primorial

Escribir `es_cuadrado(n)` con `math.isqrt`, que funciona con enteros de cualquier tamaño, y mostrar un caso en el que la versión con `math.sqrt` se equivoca. Después, calcular el primorial $p\#$ (el producto de los primos hasta $p$) con `math.prod` y comprobar el argumento de Euclides: los factores primos de $p\# + 1$ son siempre mayores que $p$.

In [8]:
def es_cuadrado(n):
    return n >= 0 and math.isqrt(n) ** 2 == n


def es_cuadrado_float(n):
    r = int(math.sqrt(n))
    return r * r == n


grande = (2**53 + 1) ** 2
print(f"¿({2**53 + 1})² es un cuadrado? isqrt: {es_cuadrado(grande)}; sqrt: {es_cuadrado_float(grande)}")
print("Cuadrados perfectos entre 1 y 10⁶:", math.isqrt(10**6))
assert es_cuadrado(grande) and not es_cuadrado_float(grande)
assert sum(es_cuadrado(k) for k in range(1, 10**6 + 1)) == math.isqrt(10**6) == 1000

¿(9007199254740993)² es un cuadrado? isqrt: True; sqrt: False
Cuadrados perfectos entre 1 y 10⁶: 1000


In [9]:
def primos_hasta(n):
    return [p for p in range(2, n + 1) if all(p % d for d in range(2, math.isqrt(p) + 1))]


def menor_factor(n):
    return next((d for d in range(2, math.isqrt(n) + 1) if n % d == 0), n)


print(f"{'p':>3} {'p#':>13} {'menor factor de p# + 1':>23}")
for p in primos_hasta(31):
    primorial = math.prod(primos_hasta(p))
    factor = menor_factor(primorial + 1)
    print(f"{p:>3} {primorial:>13} {factor:>14}{'  (primo)' if factor == primorial + 1 else ''}")
    assert factor > p

  p            p#  menor factor de p# + 1
  2             2              3  (primo)
  3             6              7  (primo)
  5            30             31  (primo)
  7           210            211  (primo)
 11          2310           2311  (primo)
 13         30030             59
 17        510510             19
 19       9699690            347
 23     223092870            317
 29    6469693230            331
 31  200560490130   200560490131  (primo)


## 8. Combinatoria, `gcd` y `lcm`

Con `math.perm`, `math.comb`, `math.factorial` y `math.prod`, contar los podios posibles en un torneo de lucha canaria con 8 equipos, las combinaciones de 6 números entre 49 y las matrículas españolas (4 cifras y 3 de 20 consonantes), comprobando los casos pequeños con `itertools`. Después, con `gcd` y `lcm` de varios argumentos, calcular cuándo vuelven a salir juntas tres guaguas que salen a las 7:00 cada 12, 18 y 30 minutos, y la mayor baldosa cuadrada que cubre sin cortes un patio de 360 × 270 cm.

In [10]:
from itertools import combinations, permutations

print("Podios con 8 equipos:", math.perm(8, 3))
print("Combinaciones de 6 entre 49:", f"{math.comb(49, 6):,}".replace(",", "."))
print("Matrículas posibles:", f"{math.prod([10] * 4 + [20] * 3):,}".replace(",", "."))
assert math.perm(8, 3) == len(list(permutations(range(8), 3))) == 336
assert math.comb(10, 4) == len(list(combinations(range(10), 4)))
assert math.comb(49, 6) == math.perm(49, 6) // math.factorial(6)

frecuencias = (12, 18, 30)
minutos = math.lcm(*frecuencias)
print(f"\nLas guaguas vuelven a coincidir a los {minutos} min: a las {7 + minutos // 60}:{minutos % 60:02d}")
assert minutos == next(t for t in count(1) if all(t % f == 0 for f in frecuencias))

lado = math.gcd(360, 270)
print(f"Baldosa de {lado} cm: {360 // lado} × {270 // lado} = {(360 // lado) * (270 // lado)} baldosas")
assert lado == max(d for d in range(1, 271) if 360 % d == 0 and 270 % d == 0)
assert math.gcd(12, 18) * math.lcm(12, 18) == 12 * 18

Podios con 8 equipos: 336
Combinaciones de 6 entre 49: 13.983.816
Matrículas posibles: 80.000.000

Las guaguas vuelven a coincidir a los 180 min: a las 10:00
Baldosa de 90 cm: 4 × 3 = 12 baldosas


## 9. Desviación muestral y poblacional, medianas y modas

Con los minutos de retraso de diez guaguas, comparar `stdev` (muestra, divide entre $n - 1$) con `pstdev` (población, divide entre $n$) y comprobar las dos con su fórmula. Con un número par de datos, comparar `median`, `median_low` y `median_high`, y con dos valores igual de frecuentes, `mode` y `multimode`.

In [11]:
import statistics as st

retrasos = [2, 5, 0, 7, 5, 3, 0, 12, 5, 0]
n, media = len(retrasos), st.fmean(retrasos)
print(f"Media {media}; stdev {st.stdev(retrasos):.4f}; pstdev {st.pstdev(retrasos):.4f}")
suma_cuadrados = sum((x - media) ** 2 for x in retrasos)
assert math.isclose(st.stdev(retrasos), math.sqrt(suma_cuadrados / (n - 1)))
assert math.isclose(st.pstdev(retrasos), math.sqrt(suma_cuadrados / n))
assert math.isclose(st.stdev(retrasos), st.pstdev(retrasos) * math.sqrt(n / (n - 1)))

print("Ordenados:", sorted(retrasos))
print(f"median {st.median(retrasos)}; median_low {st.median_low(retrasos)}; median_high {st.median_high(retrasos)}")
print(f"mode {st.mode(retrasos)}; multimode {st.multimode(retrasos)}")
assert st.median_low(retrasos) in retrasos and st.median_high(retrasos) in retrasos
assert st.multimode(retrasos) == [5, 0] and st.mode(retrasos) == 5  # en orden de primera aparición

try:
    st.stdev([4])
except st.StatisticsError as e:
    print("stdev de un solo dato:", e)

Media 3.9; stdev 3.7845; pstdev 3.5903
Ordenados: [0, 0, 0, 2, 3, 5, 5, 5, 7, 12]
median 4.0; median_low 3; median_high 5
mode 5; multimode [5, 0]
stdev de un solo dato: stdev requires at least two data points


## 10. Medias armónica y geométrica

Una guagua sube de Santa Cruz a La Laguna a 30 km/h y baja a 50 km/h por la misma carretera: calcular su velocidad media con `harmonic_mean` y comprobarla con distancia total entre tiempo total (también con tramos de distinta longitud, usando `weights`). Los visitantes de un museo cambian +12 %, −30 %, +25 % y +8 % en cuatro años: calcular el crecimiento medio anual con `geometric_mean` y ver por qué la media aritmética engaña.

In [12]:
velocidades = [30, 50]
print(f"Media aritmética {st.fmean(velocidades)} km/h; media real (armónica) {st.harmonic_mean(velocidades)} km/h")
distancia = 9
assert math.isclose(st.harmonic_mean(velocidades), 2 * distancia / sum(distancia / v for v in velocidades))

tramos = [(4, 20), (10, 60), (2, 30)]  # (km, km/h)
km = [d for d, _ in tramos]
media_tramos = st.harmonic_mean([v for _, v in tramos], weights=km)
print(f"Con tramos de {km} km: {media_tramos:.2f} km/h")
assert math.isclose(media_tramos, sum(km) / sum(d / v for d, v in tramos))

Media aritmética 40.0 km/h; media real (armónica) 37.5 km/h
Con tramos de [4, 10, 2] km: 36.92 km/h


In [13]:
cambios = [0.12, -0.30, 0.25, 0.08]
factores = [1 + c for c in cambios]
medio = st.geometric_mean(factores)
inicial = 10_000
final = inicial * math.prod(factores)
print(f"Visitantes: {inicial} -> {final:.0f}")
print(f"Crecimiento medio: geométrico {medio - 1:+.2%}, aritmético {st.fmean(cambios):+.2%}")
print(f"Con el aritmético saldrían {inicial * (1 + st.fmean(cambios)) ** 4:.0f}; con el geométrico {inicial * medio**4:.0f}")
assert math.isclose(inicial * medio ** len(cambios), final)

rng = random.Random(7)
for _ in range(100):
    datos = [rng.uniform(0.1, 100) for _ in range(8)]
    assert st.harmonic_mean(datos) <= st.geometric_mean(datos) <= st.fmean(datos)

Visitantes: 10000 -> 10584
Crecimiento medio: geométrico +1.43%, aritmético +3.75%
Con el aritmético saldrían 11587; con el geométrico 10584


## 11. La distribución normal con `NormalDist`

Suponer que el peso de un plátano sigue una normal de media 180 g y desviación 25 g. Con `statistics.NormalDist`, calcular la probabilidad de que pese menos de 150 g o entre 150 y 210 g, el peso que solo supera el 10 % (`inv_cdf`), el solapamiento con otra variedad de media 200 g y desviación 30 g, y la regla 68-95-99,7. Estimar la distribución a partir de 5000 pesos simulados con `from_samples` y marcar como atípicos los pesos nuevos con una puntuación $z$ mayor que 2,5 en valor absoluto.

In [14]:
from statistics import NormalDist

platano = NormalDist(mu=180, sigma=25)
print(f"P(peso < 150 g) = {platano.cdf(150):.4f}")
print(f"P(150 g < peso < 210 g) = {platano.cdf(210) - platano.cdf(150):.4f}")
print(f"Percentil 90: {platano.inv_cdf(0.9):.1f} g")
print(f"Solapamiento con N(200, 30): {platano.overlap(NormalDist(200, 30)):.2%}")
for k in (1, 2, 3):
    print(f"  P(|x - μ| < {k}σ) = {platano.cdf(180 + 25 * k) - platano.cdf(180 - 25 * k):.4f}")
assert platano.cdf(180) == 0.5 and math.isclose(platano.inv_cdf(platano.cdf(163)), 163)

P(peso < 150 g) = 0.1151
P(150 g < peso < 210 g) = 0.7699
Percentil 90: 212.0 g
Solapamiento con N(200, 30): 70.80%
  P(|x - μ| < 1σ) = 0.6827
  P(|x - μ| < 2σ) = 0.9545
  P(|x - μ| < 3σ) = 0.9973


In [15]:
rng = random.Random(2026)
muestra = [rng.gauss(180, 25) for _ in range(5_000)]
estimada = NormalDist.from_samples(muestra)
print(f"Estimada: μ = {estimada.mean:.2f} g, σ = {estimada.stdev:.2f} g")
assert abs(estimada.mean - 180) < 1.5 and abs(estimada.stdev - 25) < 1.5

nuevos = [176, 241, 98, 205, 119, 300]
print(f"{'peso':>5} {'z':>6}")
for peso in nuevos:
    z = estimada.zscore(peso)
    print(f"{peso:>5} {z:>6.2f}{'  atípico' if abs(z) > 2.5 else ''}")
atipicos = [p for p in nuevos if abs(estimada.zscore(p)) > 2.5]
assert atipicos == [p for p in nuevos if abs(p - estimada.mean) / estimada.stdev > 2.5] == [98, 300]

Estimada: μ = 179.94 g, σ = 24.78 g
 peso      z
  176  -0.16
  241   2.46
   98  -3.31  atípico
  205   1.01
  119  -2.46
  300   4.85  atípico


## 12. Un generador congruencial lineal

Sin usar `random`, escribir la clase `GeneradorLCG`, que produzca la secuencia $x_{k+1} = (a\,x_k + c) \bmod m$ con $a = 1\,664\,525$, $c = 1\,013\,904\,223$ y $m = 2^{32}$, y un método `entero(minimo, maximo)` para enteros de 0 a 100. Comprobar que la misma semilla repite la secuencia, ver por qué no hay que usar los bits bajos, medir el periodo de un generador con $m = 2^8$ y hacer un test de uniformidad $\chi^2$ con 101 000 números.

In [16]:
class GeneradorLCG:
    def __init__(self, semilla, a=1_664_525, c=1_013_904_223, m=2**32):
        self.a, self.c, self.m = a, c, m
        self.estado = semilla % m

    def siguiente(self):
        self.estado = (self.a * self.estado + self.c) % self.m
        return self.estado

    def aleatorio(self):
        return self.siguiente() / self.m

    def entero(self, minimo, maximo):
        # Escalar usa los bits altos, que son los de mayor calidad en un LCG
        return minimo + int(self.aleatorio() * (maximo - minimo + 1))


g = GeneradorLCG(semilla=1977)
print("Diez enteros de 0 a 100:", [g.entero(0, 100) for _ in range(10)])
assert [GeneradorLCG(42).entero(0, 100) for _ in range(5)] == [GeneradorLCG(42).entero(0, 100) for _ in range(5)]

g = GeneradorLCG(semilla=1977)
print("Bit bajo de cada estado:", [g.siguiente() % 2 for _ in range(12)])


def periodo(generador):
    inicial, pasos = generador.estado, 0
    while True:
        pasos += 1
        if generador.siguiente() == inicial:
            return pasos


print("Periodo con m = 2⁸:", periodo(GeneradorLCG(5, m=2**8)))

Diez enteros de 0 a 100: [0, 60, 49, 85, 99, 85, 50, 88, 23, 15]
Bit bajo de cada estado: [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
Periodo con m = 2⁸: 256


In [17]:
from collections import Counter


def chi_cuadrado(generador, n=101_000, valores=101):
    frecuencias = Counter(generador.entero(0, valores - 1) for _ in range(n))
    esperada = n / valores
    return sum((frecuencias[v] - esperada) ** 2 / esperada for v in range(valores)), frecuencias


CRITICO = 124.34  # χ² con 100 grados de libertad al 5 %
for nombre, generador in (("m = 2³²", GeneradorLCG(2026)), ("m = 2⁸", GeneradorLCG(2026, m=2**8))):
    chi2, frecuencias = chi_cuadrado(generador)
    print(f"{nombre:<8} χ² = {chi2:>9.1f}  mín {min(frecuencias.values())}, máx {max(frecuencias.values())}"
          f"  -> {'parece uniforme' if chi2 < CRITICO else 'no es uniforme'}")
    if generador.m == 2**32:
        assert chi2 < CRITICO and sorted(frecuencias) == list(range(101))
    else:
        assert chi2 > CRITICO

m = 2³²  χ² =     102.3  mín 923, máx 1084  -> parece uniforme

m = 2⁸   χ² =    3906.7  mín 788, máx 1185  -> no es uniforme


## 13. `randrange` con paso, `secrets` y una fecha aleatoria

Con un `random.Random` con semilla, sacar múltiplos de 5 entre 0 y 100 con `randrange` y un paso. Generar con `secrets` un token hexadecimal y otro apto para URL de 16 bytes, mostrando solo su longitud y su alfabeto. Escribir `fecha_aleatoria(inicio, fin, rng)`, que elija una fecha entre dos (ambas incluidas), y comprobar que puede devolver las dos fechas extremas.

In [18]:
import secrets
import string
from datetime import date, timedelta

rng = random.Random(2026)
multiplos = [rng.randrange(0, 101, 5) for _ in range(12)]
print("Múltiplos de 5:", multiplos)
assert all(x % 5 == 0 and 0 <= x <= 100 for x in multiplos)
assert {rng.randrange(1, 10, 2) for _ in range(200)} == {1, 3, 5, 7, 9}

hexadecimal, para_url = secrets.token_hex(16), secrets.token_urlsafe(16)
print(f"token_hex(16): {len(hexadecimal)} caracteres; token_urlsafe(16): {len(para_url)} caracteres")
assert len(hexadecimal) == 32 and set(hexadecimal) <= set("0123456789abcdef")
assert len(para_url) == 22 and set(para_url) <= set(string.ascii_letters + string.digits + "-_")


def fecha_aleatoria(inicio, fin, rng=random):
    return inicio + timedelta(days=rng.randrange((fin - inicio).days + 1))


fechas = sorted(fecha_aleatoria(date(2026, 1, 1), date(2026, 12, 31), rng) for _ in range(5))
print("Fechas de 2026:", [f.isoformat() for f in fechas])
corto = {fecha_aleatoria(date(2026, 5, 28), date(2026, 5, 30), rng) for _ in range(200)}
assert corto == {date(2026, 5, 28), date(2026, 5, 29), date(2026, 5, 30)}

Múltiplos de 5: [15, 50, 80, 80, 100, 15, 35, 95, 95, 85, 65, 90]
token_hex(16): 32 caracteres; token_urlsafe(16): 22 caracteres
Fechas de 2026: ['2026-01-18', '2026-01-24', '2026-03-30', '2026-04-11', '2026-11-28']


## 14. Cantidad económica de pedido

La cantidad que conviene pedir de cada vez para minimizar los costes de una tienda es $Q = \sqrt{\dfrac{2\,C\,D}{H}}$, con $D$ la demanda anual, $C$ el coste fijo de cada pedido y $H$ el coste de almacenar una unidad durante un año. Calcular $Q$, los pedidos al año y el coste anual de cada producto de una lista, y comprobar que con $Q$ el coste de pedir iguala al de almacenar y que ningún otro tamaño de pedido entero es más barato.

In [19]:
productos = [  # (nombre, D unidades/año, C € por pedido, H € por unidad y año)
    ("Gofio (kg)", 2_400, 40, 1.5),
    ("Mojo picón (frasco)", 1_800, 25, 0.8),
    ("Queso (pieza)", 600, 60, 4.0),
    ("Plátanos (caja)", 5_200, 35, 2.5),
]


def cantidad_optima(D, C, H):
    return math.sqrt(2 * C * D / H)


def coste_anual(Q, D, C, H):
    return C * D / Q + H * Q / 2


print(f"{'Producto':<20} {'Q':>7} {'pedidos/año':>12} {'coste €/año':>12}")
for nombre, D, C, H in productos:
    Q = cantidad_optima(D, C, H)
    print(f"{nombre:<20} {Q:>7.1f} {D / Q:>12.1f} {coste_anual(Q, D, C, H):>12.2f}")
    assert math.isclose(C * D / Q, H * Q / 2)
    mejor_entero = min(range(1, 3 * math.ceil(Q)), key=lambda q: coste_anual(q, D, C, H))
    assert mejor_entero in (math.floor(Q), math.ceil(Q))

Producto                   Q  pedidos/año  coste €/año
Gofio (kg)             357.8          6.7       536.66
Mojo picón (frasco)    335.4          5.4       268.33
Queso (pieza)          134.2          4.5       536.66
Plátanos (caja)        381.6         13.6       953.94


## 15. Derivar e integrar un polinomio con `sympy`

Derivar e integrar simbólicamente $p(x) = 3x^4 - 2x^3 + 5x - 7$ con `sympy`, comprobar que la derivada de la integral devuelve $p$, calcular $\int_0^2 p(x)\,dx$ y contrastar la derivada en un punto con una diferencia finita. Escribir además `derivar` e `integrar` sobre la lista de coeficientes (regla de la potencia, con `Fraction`) y comprobar que coinciden con `sympy`.

In [20]:
from fractions import Fraction

import sympy as sp

x = sp.symbols("x")
p = 3 * x**4 - 2 * x**3 + 5 * x - 7
derivada = sp.diff(p, x)
integral = sp.integrate(p, x)
print("p'(x) =", derivada)
print("∫ p(x) dx =", integral, "+ C")
print("∫₀² p(x) dx =", sp.integrate(p, (x, 0, 2)))
assert sp.expand(sp.diff(integral, x) - p) == 0
assert sp.integrate(p, (x, 0, 2)) == integral.subs(x, 2) - integral.subs(x, 0)

f, h = sp.lambdify(x, p), 1e-5
aproximada = (f(1.5 + h) - f(1.5 - h)) / (2 * h)
print(f"p'(1,5): exacta {float(derivada.subs(x, 1.5))}, diferencia finita {aproximada:.6f}")
assert math.isclose(aproximada, float(derivada.subs(x, 1.5)), rel_tol=1e-8)

p'(x) = 12*x**3 - 6*x**2 + 5
∫ p(x) dx = 3*x**5/5 - x**4/2 + 5*x**2/2 - 7*x + C
∫₀² p(x) dx = 36/5


p'(1,5): exacta 32.0, diferencia finita 32.000000


In [21]:
def derivar(coeficientes):
    """Coeficientes de menor a mayor grado: [a0, a1, a2, ...]."""
    return [k * a for k, a in enumerate(coeficientes)][1:]


def integrar(coeficientes):
    return [Fraction(0)] + [Fraction(a, k + 1) for k, a in enumerate(coeficientes)]


coefs = [-7, 5, 0, -2, 3]
print("derivar:", derivar(coefs))
print("integrar:", [str(c) for c in integrar(coefs)])


def a_sympy(coeficientes):
    return sum(sp.Rational(a) * x**k for k, a in enumerate(coeficientes))


assert a_sympy(coefs) == p
assert sp.expand(a_sympy(derivar(coefs)) - derivada) == 0
assert sp.expand(a_sympy(integrar(coefs)) - integral) == 0
assert derivar(integrar(coefs)) == coefs

derivar: [5, 0, -6, 12]
integrar: ['0', '-7', '5/2', '0', '-1/2', '3/5']
